# 特征重要性：哪些特征最有用
> 难度标签：中级 | 预计时长：10-20分钟 | 前置知识：无学习经验


> 所属模块：11_特征工程 | 源文件：特征重要性.py | 核心功能：树模型特征重要性、Permutation Importance、SHAP

## 概述

特征重要性帮助理解模型和数据。三种方法：基于不纯度（树模型内置）、Permutation Importance（打乱特征看性能下降）、SHAP 值（最可靠）。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## 数学原理

### 1. 树模型的不纯度减少（Gini Importance）

决策树在每个节点通过特征 $j$ 的分裂减少不纯度。特征 $j$ 的总重要性是所有节点中用 $j$ 分裂带来的不纯度减少之和：

$$\text{Imp}(j) = \sum_{\text{nodes } t \text{ using } j} \left[\frac{n_t}{n} \left(I(t) - \frac{n_{tL}}{n_t}I(t_L) - \frac{n_{tR}}{n_t}I(t_R)\right)\right]$$

**Gini 不纯度**：$I_G(t) = 1 - \sum_c p_c^2$

**信息熵**：$I_E(t) = -\sum_c p_c \log p_c$

归一化后得到 `feature_importances_`：

$$\hat{f}_j = \frac{\text{Imp}(j)}{\sum_{k}\text{Imp}(k)}$$

随机森林对所有树取平均：$\hat{f}_j = \frac{1}{B}\sum_{b=1}^{B}\hat{f}_j^{(b)}$

### 2. 排列重要性（Permutation Importance）

不依赖模型类型，通过打乱特征值衡量重要性：

$$\text{PI}(j) = s(\text{perm}(X_j), y) - s(X, y)$$

其中 $s$ 是评估指标（如准确率），$\text{perm}(X_j)$ 是随机打乱第 $j$ 列后的数据。

- $\text{PI}(j) > 0$：打乱后性能下降，特征 $j$ 重要
- $\text{PI}(j) \approx 0$：特征 $j$ 不重要
- $\text{PI}(j) < 0$：可能是噪声特征或过拟合

### 3. 相关性分析

**皮尔逊相关系数**（线性关系）：

$$r_j = \frac{\sum_{i}(x_{ij} - \bar{x}_j)(y_i - \bar{y})}{\sqrt{\sum_i(x_{ij}-\bar{x}_j)^2 \cdot \sum_i(y_i-\bar{y})^2}}$$

$|r_j|$ 越大，特征与目标的线性关系越强。

### 4. 不纯度重要性的偏差

Gini Importance 对高基数特征有偏好（连续特征或类别多的特征更容易被选中分裂）。Permutation Importance 无此偏差。

### 5. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `rf.feature_importances_` | $\hat{f}_j = \frac{1}{B}\sum_b \text{Imp}^{(b)}(j) / \sum_k \text{Imp}^{(b)}(k)$ |
| `permutation_importance(model, X, y)` | $\text{PI}(j) = s(\text{perm}(X_j), y) - s(X, y)$ |
| `np.corrcoef(X[:, j], y)[0,1]` | 皮尔逊相关系数 $r_j$ |
| `np.argsort(importances)[::-1]` | 按重要性降序排列特征索引 |

### 1. 树模型内置特征重要性

分析各特征对模型预测的贡献度。

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

print("=" * 60)
print("1. 树模型内置特征重要性 (feature_importances_)")
print("=" * 60)

X_cls, y_cls = make_classification(
    n_samples=500, n_features=12, n_informative=5,
    n_redundant=2, n_classes=2, random_state=42
)

feature_names = [f"特征{i}" for i in range(X_cls.shape[1])]

# 随机森林
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_cls, y_cls)

print("随机森林 feature_importances_:")
importance_rf = rf.feature_importances_
for name, imp in sorted(zip(feature_names, importance_rf), key=lambda x: -x[1]):
    bar = "#" * int(imp * 50)
    print(f"  {name}: {imp:.4f} {bar}")

# 梯度提升
gb = GradientBoostingClassifier(n_estimators=200, random_state=42)
gb.fit(X_cls, y_cls)

print("\n梯度提升 feature_importances_:")
importance_gb = gb.feature_importances_
for name, imp in sorted(zip(feature_names, importance_gb), key=lambda x: -x[1]):
    bar = "#" * int(imp * 50)
    print(f"  {name}: {imp:.4f} {bar}")

### 2. 排列重要性 (Permutation Importance)

运行 2. 排列重要性 (Permutation Importance) 的代码，观察算法在该环节的行为。

In [ ]:
from sklearn.inspection import permutation_importance

print("\n" + "=" * 60)
print("2. 排列重要性 (Permutation Importance)")
print("=" * 60)

X_train, X_test, y_train, y_test = train_test_split(
    X_cls, y_cls, test_size=0.3, random_state=42
)

rf2 = RandomForestClassifier(n_estimators=100, random_state=42)
rf2.fit(X_train, y_train)

# 排列重要性：打乱每个特征的值，观察准确率下降多少
perm_result = permutation_importance(
    rf2, X_test, y_test, n_repeats=10, random_state=42
)

print("排列重要性 (测试集上，重复10次):")
print(f"  {'特征':<8} {'均值':>8} {'标准差':>8}")
print(f"  {'-'*28}")
for name, mean_imp, std_imp in sorted(
    zip(feature_names, perm_result.importances_mean, perm_result.importances_std),
# --- 赋值/配置 ---
    key=lambda x: -x[1]
):
    print(f"  {name:<8} {mean_imp:>8.4f} {std_imp:>8.4f}")

### 3. 相关性分析

运行 3. 相关性分析 的代码，观察算法在该环节的行为。

In [ ]:
print("\n" + "=" * 60)
print("3. 相关性分析 (Pearson相关系数)")
print("=" * 60)

# 生成回归数据
X_reg, y_reg = make_regression(
    n_samples=300, n_features=8, n_informative=4,
    noise=10, random_state=42
)

# 计算Pearson相关系数
correlations = np.array([
    np.corrcoef(X_reg[:, i], y_reg)[0, 1]
    for i in range(X_reg.shape[1])
])

print("各特征与目标变量的Pearson相关系数:")
print(f"  {'特征':<8} {'相关系数':>10} {'|相关|':>8}")
print(f"  {'-'*30}")
for name, corr in sorted(
    zip([f"特征{i}" for i in range(X_reg.shape[1])], correlations),
# --- 继续 ---
    key=lambda x: -abs(x[1])
):
    strength = "强" if abs(corr) > 0.5 else "中" if abs(corr) > 0.3 else "弱"
    print(f"  {name:<8} {corr:>10.4f} {abs(corr):>8.4f} ({strength})")

### 4. Spearman等级相关 (非线性关系)

运行 4. Spearman等级相关 (非线性关系) 的代码，观察算法在该环节的行为。

In [ ]:
from scipy.stats import spearmanr

print("\n" + "=" * 60)
print("4. Spearman等级相关 (非线性关系)")
print("=" * 60)

# 创建非线性关系: y = x^2 + 噪声
np.random.seed(42)
x_nonlinear = np.random.uniform(-3, 3, 200)
y_nonlinear = x_nonlinear ** 2 + np.random.normal(0, 0.5, 200)

pearson_corr = np.corrcoef(x_nonlinear, y_nonlinear)[0, 1]
spearman_corr, spearman_p = spearmanr(x_nonlinear, y_nonlinear)

print(f"Pearson相关系数 (线性):  {pearson_corr:.4f}")
print(f"Spearman相关系数 (单调): {spearman_corr:.4f}")
print(f"Spearman p值:            {spearman_p:.6f}")
print("说明: 当变量间存在非线性单调关系时，Spearman比Pearson更敏感")

### 5. 综合对比

对比不同模型或配置的性能差异。

In [ ]:
print("\n" + "=" * 60)
print("5. 三种方法的对比")
print("=" * 60)

# 对齐排名
rf_rank = np.argsort(-importance_rf)
perm_rank = np.argsort(-perm_result.importances_mean)
corr_rank = np.argsort(-np.abs(correlations))

print(f"{'特征':<8} {'RF重要性排名':>12} {'排列重要性排名':>14} {'相关系数排名':>12}")
print(f"{'-'*50}")
for i in range(X_cls.shape[1]):
    name = feature_names[i]
    rf_r = np.where(rf_rank == i)[0][0] + 1
# --- 数值计算 ---
    perm_r = np.where(perm_rank == i)[0][0] + 1
    # 树模型和相关性维度不同，这里只展示树模型的
    print(f"  {name:<8} {rf_r:>12} {perm_r:>14}")

print("\n总结:")
print("  - 树模型重要性: 基于训练集，可能偏向高基数特征")
print("  - 排列重要性: 基于测试集，更可靠的泛化评估")
print("  - 相关性分析: 简单快速，但只能捕捉线性/单调关系")

## 关键代码解释

```python
from sklearn.inspection import permutation_importance
result = permutation_importance(model, X_test, y_test, n_repeats=10)
```

## 注意事项

1. 不纯度重要性对高基数特征有偏
2. Permutation Importance 更可靠但更慢
3. SHAP 值可以解释单个预测

## 总结与延伸

以上代码展示了 **特征重要性** 的完整流程。

**解读要点**：
- 关注 **特征重要性** 排名和分布
- 对比特征选择前后的模型性能
- 观察特征交叉是否带来性能提升

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **SHAP**：统一的特征归因框架
- **LIME**：局部可解释性
- **特征重要性的稳定性**：不同随机种子结果可能不同
